# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [7]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [8]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
ENTERTAINMENT = {
    "manga": (
        "Omniscient reader's viewpoint is a manga inspired from a korean web novel. It beautifully depicts the mc's connection with the novel's characters in the story ."
        "The greatest estate developer is a manga revolved mostly around a modern-day civil engineering student who finds himself in a novel he read. He brings a lot of mind-blowing changes in the debt-ridden estate and saves it."
        "Wind Breaker manga revolves around a character who only knows to study and cycle. He makes new friends and joins a tournament. What started as a fun hobby ends up enlightening him about his uncle's past"
        "School bus graveyard is a thrilling webtoon that depicts a high-schooler who has a weird connection with beings from a different dimensions. She and her new group of friends must find ways to survive each night as they find themselves thrown into the phantom dimension."
    ),
    "novels":(

        "A good girl's guide to murder is a murder-thriller in which a student who simply takes on a case for her school project, ends up finding real clues about the missing people. She along with her friend solve the murder case and bring peace to their small town."
        "Never lie is a thriller by freida in which a couple gets stranded in a creepy house belonging to a dead person during house-hunting. Turns out both of them have secrets they don't want the other to know."
        "The empyrean series revolves around a fragile girl who has a rather strong heart and mind. She goes through a lot of things from bonding dragons to losing loved ones during her journey in the war college. "
    ),

    "shows/dramas":(
        "Nightmare teacher is a kdrama is a thriller which is about a teacher who knows more than he is letting on."
        "TwentyfiveTwentyone is a high school kdrama about fencing. Two friend end up on the opposing teams with a lot of weight on their shoulders. We still see them grow and navigate their love lives."
        "100 days of deception is about a girl who doesnt want anything to do with the war going on around her and just wants to go to america. She ends up working as a spy for korean independence and starts spying on the japanese."
    ),

}

In [9]:
# @title Define a tool
def search_entertainment(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in ENTERTAINMENT.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant information found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_entertainment("minimum of 4"))


[Source: shows/dramas]
Nightmare teacher is a kdrama is a thriller which is about a teacher who knows more than he is letting on.TwentyfiveTwentyone is a high school kdrama about fencing. Two friend end up on the opposing teams with a lot of weight on their shoulders. We still see them grow and navigate their love lives.100 days of deception is about a girl who doesnt want anything to do with the war going on around her and just wants to go to america. She ends up working as a spy for korean independence and starts spying on the japanese.

[Source: novels]
A good girl's guide to murder is a murder-thriller in which a student who simply takes on a case for her school project, ends up finding real clues about the missing people. She along with her friend solve the murder case and bring peace to their small town.Never lie is a thriller by freida in which a couple gets stranded in a creepy house belonging to a dead person during house-hunting. Turns out both of them have secrets they don't

In [10]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    # This will break.
    # TODO: Find the right model to call in this cell from developer docs.
    model_name="claude",
    tools=[search_entertainment],
    system_instruction=(
        '''
        You are a professional reviewer for various entertainment shows, novels and comics. Recommend me the best sources to read or watch based on my preferences.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/claude


In [14]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_entertainment",
        description="Search the entertainment records to find most the relevant source.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model="gemini-3.8-flash",
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_entertainment(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("Recommend me engineering related comics")

Question: Recommend me engineering related comics
--------------------------------------------------
Step 1: model calls search_entertainment({'query': 'engineering comics'})
  Result preview: [Source: manga]
Omniscient reader's viewpoint is a manga inspired from a korean web novel. It beauti...

Final answer:
Here are some of the best engineering-related comics, spanning webtoons, manga, webcomics, and graphic novels across different disciplines:

---

### **1. Manhwa & Manga**

* **The Greatest Estate Developer** *(Civil / Architectural Engineering)*
  * **Format:** Manhwa / Webtoon
  * **Premise:** A modern civil engineering student is transported into the body of an indebted, useless noble in a fantasy novel. To avoid ruin, he relies on his textbook engineering knowledge—soil mechanics, surveying, thermal heating (ondol), irrigation canals, bridge building, and reinforced masonry—to completely modernize the estate. It is intensely funny, accurate with its core engineering concepts,

In [25]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "Whats the manga which is based on a charcater cycling and finding about his uncle?",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
    "Which drama has a sports component like cyccling , fencing etc.?"
    "Suggest me a few thriller novles."
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: Whats the manga which is based on a charcater cycling and finding about his uncle?
--------------------------------------------------
  Rate limit hit, waiting 20s...
  Rate limit hit, waiting 40s...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 16h15.9919257s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https:/

In [21]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)

AssertionError: 